# 01 — Model setup: iDT1294 for PHB optimization

This initial notebook loads the genome-scale metabolic model iDT1294 of Rhodopseudomonas palustris, developed by Tec-Campos et al. (2023), using the COBRApy framework.

Input: model in .mat format

Output: model in .xml format format

# Installation

In [1]:
# Install cobrapy
!pip install -qq cobra catboost

In [2]:
# Check installation
!pip show cobra

Name: cobra
Version: 0.30.0
Summary: COBRApy is a package for constraint-based modeling of metabolic networks.
Home-page: https://opencobra.github.io/cobrapy
Author: The cobrapy core development team.
Author-email: cobra-pie@googlegroups.com
License: LGPL-2.0-or-later OR GPL-2.0-or-later
Location: /usr/local/lib/python3.12/dist-packages
Requires: appdirs, depinfo, diskcache, future, httpx, numpy, optlang, pandas, pydantic, python-libsbml, rich, ruamel.yaml, swiglpk
Required-by: 


# Mount drive

In [3]:
import os
from pathlib import Path
from google.colab import drive

def mount_drive():
  drive.mount('/content/drive', force_remount=True)
  drive_folder = "metabolic_modelling/phb-optimization-rpalustris/"
  os.chdir('/content/drive/MyDrive/'+ drive_folder)
  global PROJECT_ROOT
  PROJECT_ROOT = Path(os.getcwd())

mount_drive()


Mounted at /content/drive


# Load packages

In [4]:
# load packages

import logging
import cobra
from cobra import Reaction, Metabolite, Model, io
from cobra.io import (
    load_model, load_json_model, save_json_model,
    load_matlab_model, save_matlab_model,
    read_sbml_model, write_sbml_model
)

from cobra.flux_analysis import flux_variability_analysis, pfba

from cobra.util.solver import linear_reaction_coefficients

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
from itertools import product
import seaborn as sns
import json
import math
import copy
from tqdm import tqdm


# Set paths

In [5]:
# Import path saved in src

import sys
sys.path.append(str(PROJECT_ROOT / "src"))
from src.paths import MODELS_DIR, PHB_MODEL_DIR, PHB_CHECKPOINTS_DIR


# Load .mat model

In [7]:
# Load two versions of model iDT1294 in matlab format for comparison

# iDT1294
model_iDT1294_matlab_path = MODELS_DIR / "iDT1294_originals" / "iDT1294.mat"
model_base = load_matlab_model(str(model_iDT1294_matlab_path))

# iDT1294 PHOTO
model_iDT1294_photo_matlab_path = MODELS_DIR / "iDT1294_originals" / "iDT1294Photo.mat"
model_photo = load_matlab_model(str(model_iDT1294_matlab_path))

print("Models loaded successfully")

Models loaded successfully


# Check default model tec_campos_iDT1294

In [8]:
# Load model
model_base

Name,iDT1294
Memory address,789000279df0
Number of metabolites,2124
Number of reactions,2721
Number of genes,1294
Number of groups,118
Objective expression,1.0*BIOMASS__1 - 1.0*BIOMASS__1_reverse_063c7
Compartments,"c, u, p, e"


In [9]:
# Check model summary
model_base.summary() # BIOMASS__1 is the default objective function

Metabolite,Reaction,Flux,C-Number,C-Flux
actn__R_e,EX_actn__R_e,0.6784,4,12.28%
ca2_e,EX_ca2_e,1.03E-05,0,0.00%
cinnm_e,EX_cinnm_e,2.153,9,87.72%
cl_e,EX_cl_e,2.06E-05,0,0.00%
cobalt2_e,EX_cobalt2_e,4.533E-06,0,0.00%
fe3_e,EX_fe3_e,0.0337,0,0.00%
k_e,EX_k_e,1.366E-06,0,0.00%
mg2_e,EX_mg2_e,0.01097,0,0.00%
mn2_e,EX_mn2_e,1.554E-07,0,0.00%
na1_e,EX_na1_e,1.051E-07,0,0.00%


In [10]:
# Check model compartments
model_base.compartments

{'c': 'c', 'u': 'u', 'p': 'p', 'e': 'e'}

## PHB synthesis reaction

In [11]:
# Check reaction directly associated to PHB synthesis
phb_rxn = model_base.reactions.get_by_id("BIOMASS__1")
print(phb_rxn.bounds)
print(phb_rxn.reaction)


(0.0, 2.0)
30.0 atp_c + 0.0977 bm_carbs_c + 0.00119 bm_cofactors_c + 0.0795 bm_cw_c + 0.0073 bm_dna_c + 0.01 bm_min_c + 0.159 bm_oth_c + 0.0197 bm_pigm_c + 0.5112 bm_pro_c + 0.1136 bm_rna_c + 30.0 h2o_c --> 30.0 adp_c + 30.0 h_c + 30.0 pi_c


# Cap lower and upper bounds

In [12]:
# Cap extreme bounds (values <>1000)

def cap_bounds(model, lb_limit=-1000, ub_limit=1000):
    for rxn in model.reactions:
        if rxn.lower_bound < lb_limit:
            rxn.lower_bound = lb_limit
        if rxn.upper_bound > ub_limit:
            rxn.upper_bound = ub_limit

# Apply to both models
cap_bounds(model_base)
cap_bounds(model_photo)

# Set BIOMASS__1 bounds


In [13]:
# Simulate PHB production decoupled from biomass

# For model_base
rxn_biomass = model_base.reactions.get_by_id("BIOMASS__1")
rxn_biomass.lower_bound = 0
rxn_biomass.upper_bound = 0


print("model_base BIOMASS__1 bounds:", rxn_biomass.bounds)
print("model_base BIOMASS__1 reaction:", rxn_biomass.reaction)

# For model_photo
rxn_biomass_photo = model_photo.reactions.get_by_id("BIOMASS__1")
rxn_biomass_photo.lower_bound = 0
rxn_biomass_photo.upper_bound = 0


print("model_photo BIOMASS__1 bounds:", rxn_biomass_photo.bounds)
print("model_photo BIOMASS__1 reaction:", rxn_biomass_photo.reaction)

model_base BIOMASS__1 bounds: (0, 0)
model_base BIOMASS__1 reaction: 30.0 atp_c + 0.0977 bm_carbs_c + 0.00119 bm_cofactors_c + 0.0795 bm_cw_c + 0.0073 bm_dna_c + 0.01 bm_min_c + 0.159 bm_oth_c + 0.0197 bm_pigm_c + 0.5112 bm_pro_c + 0.1136 bm_rna_c + 30.0 h2o_c --> 30.0 adp_c + 30.0 h_c + 30.0 pi_c
model_photo BIOMASS__1 bounds: (0, 0)
model_photo BIOMASS__1 reaction: 30.0 atp_c + 0.0977 bm_carbs_c + 0.00119 bm_cofactors_c + 0.0795 bm_cw_c + 0.0073 bm_dna_c + 0.01 bm_min_c + 0.159 bm_oth_c + 0.0197 bm_pigm_c + 0.5112 bm_pro_c + 0.1136 bm_rna_c + 30.0 h2o_c --> 30.0 adp_c + 30.0 h_c + 30.0 pi_c


# Set PHBS_syn as objective function

In [14]:
# For model_base
rxn_phb = model_base.reactions.get_by_id("PHBS_syn")
rxn_phb.lower_bound = 0
rxn_phb.upper_bound = 0.3978
model_base.objective = rxn_phb

print("model_base PHBS_syn bounds:", rxn_phb.bounds)
print("model_base PHBS_syn reaction:", rxn_phb.reaction)

# For model_photo
rxn_phb_photo = model_photo.reactions.get_by_id("PHBS_syn")
rxn_phb_photo.lower_bound = 0
rxn_phb_photo.upper_bound = 0.3978
model_photo.objective = rxn_phb_photo

print("model_photo PHBS_syn bounds:", rxn_phb_photo.bounds)
print("model_photo PHBS_syn reaction:", rxn_phb_photo.reaction)

model_base PHBS_syn bounds: (0, 0.3978)
model_base PHBS_syn reaction: 3hbcoa__R_c + phbg_c --> PHB_c + coa_c
model_photo PHBS_syn bounds: (0, 0.3978)
model_photo PHBS_syn reaction: 3hbcoa__R_c + phbg_c --> PHB_c + coa_c


# Compare metabolites and reactions between models

In [15]:
# --- Compare reactions ---
rxn_base = set(r.id for r in model_base.reactions)
rxn_photo = set(r.id for r in model_photo.reactions)

only_in_base_rxns = rxn_base - rxn_photo
only_in_photo_rxns = rxn_photo - rxn_base

print("Reactions only in base model:", only_in_base_rxns)
print("Reactions only in photo model:", only_in_photo_rxns)

# --- Compare bounds of common reactions ---
common_rxns = rxn_base & rxn_photo
bounds_diff = {}

for rxn_id in common_rxns:
    r_base = model_base.reactions.get_by_id(rxn_id)
    r_photo = model_photo.reactions.get_by_id(rxn_id)
    if r_base.lower_bound != r_photo.lower_bound or r_base.upper_bound != r_photo.upper_bound:
        bounds_diff[rxn_id] = {
            "base_lb": r_base.lower_bound,
            "base_ub": r_base.upper_bound,
            "photo_lb": r_photo.lower_bound,
            "photo_ub": r_photo.upper_bound
        }
print("\nReactions with different bounds after capping:")
if not bounds_diff:
    print("Both models have the same metabolites, reactions, lower bounds and upper bounds")
else:
    for rxn, b in bounds_diff.items():
        print(f"{rxn}: Base LB={b['base_lb']}, UB={b['base_ub']} | Photo LB={b['photo_lb']}, UB={b['photo_ub']}")

Reactions only in base model: set()
Reactions only in photo model: set()

Reactions with different bounds after capping:
Both models have the same metabolites, reactions, lower bounds and upper bounds


# Save reactions and metabolites for reference

In [16]:
# -----------------------
# Save reactions table
# -----------------------
rxn_data = []

for r in model_base.reactions:
    rxn_data.append({
        "id": r.id,
        "name": r.name,
        "equation": r.reaction,
        "lower_bound": r.lower_bound,
        "upper_bound": r.upper_bound,
        "genes": ";".join(g.id for g in r.genes),
    })

rxn_df = pd.DataFrame(rxn_data)
rxn_df.to_csv(PHB_CHECKPOINTS_DIR / "01_model_phb_reactions.csv", index=False)

# -----------------------
# Save metabolites table
# -----------------------
met_data = []

for m in model_base.metabolites:
    met_data.append({
        "id": m.id,
        "name": m.name,
        "formula": m.formula,
        "compartment": m.compartment,
        "charge": m.charge,
        "in_reactions": len(m.reactions),
    })

met_df = pd.DataFrame(met_data)
met_df.to_csv(PHB_CHECKPOINTS_DIR / "01_model_phb_metabolites.csv", index=False)


# Save medium of phb base model

GEMs simulate media through the reactions staring with "EX" with negative flow. These reactions are saved next.

In [19]:
# Check default medium conditions
model_base.medium

# Convert to a DataFrame
medium_df = pd.DataFrame(list(model_base.medium.items()), columns=["Reaction", "UptakeBound"])

# Save to Excel
medium_df.to_csv(PHB_CHECKPOINTS_DIR/ "01_model_phb_medium.csv", index=False)

# Print the medium elements
#for rxn_id, bound in model.medium.items():
#   print(rxn_id, bound)



# Save base model

This model has PHBS_syn as objective reaction, BIOMASS__1 reaction was set to zero to simulate uncoupled PHB production from growth, and lower and upper bounds are capped to max values of 1000, which are the standard values in GEMs. Only the resulting base model is saved as a .xml file

In [17]:
# Since both models are the same, only the base model is saved below

# Target file

phb_model_base_file = PHB_MODEL_DIR / "01_model_rpalustris_PHB.xml"

# Save COBRA model as .xml file
cobra.io.write_sbml_model(model_base, str(phb_model_base_file))
